In [0]:
dbutils.widgets.dropdown(name="environment", defaultValue="dev",
                         choices=["dev","qa","prd"], label="select Environment")
env = dbutils.widgets.get("environment")
print(f"Environment: {env}")

catalog = f"saleslake_{env}"
bronzeSchema = f"bronze_schema_{env}"

targetTable    = f"{catalog}.{bronzeSchema}.rawRegion"
srcFileLoc  = f"/Volumes/{catalog}/{bronzeSchema}/vol_saleslake_src_files_{env}/daily_region/"
print(f"Target table : {targetTable}")
print(f"Source files : {srcFileLoc}")


In [0]:
spark.sql(f"""
COPY INTO {targetTable}
FROM (
    SELECT region_id, region_code, region_name, country, sub_region, regional_manager, created_date,
           current_timestamp() AS ingest_ts
    FROM "{srcFileLoc}"
)
FILEFORMAT = CSV
FORMAT_OPTIONS ('header' = 'true')
COPY_OPTIONS  ('mergeSchema' = 'false')
""")

print(f"Bronze load complete for {targetTable}")


In [0]:
spark.sql(f"SELECT COUNT(*) AS row_count FROM {targetTable}").display()